# Setup

In [1]:
from langchain_openai import OpenAI
from langchain.cache import InMemoryCache,SQLiteCache
from langchain.globals import set_llm_cache
import os
import json
import hashlib
import yaml

In [2]:
with open('config.yaml', 'r') as config_file:
    config = yaml.safe_load(config_file)
os.environ['OPENAI_API_KEY'] = config['OPENAI_API_KEY']

# Memória

In [3]:
openai = OpenAI(model_name='gpt-3.5-turbo-instruct')
set_llm_cache(InMemoryCache())

In [4]:
prompt = 'Me diga em poucas palavras que foi Carl Sagan.'
response1 = openai.invoke(prompt)
print("Primeira resposta (API chamada):", response1)

Primeira resposta (API chamada): 

Carl Sagan foi um renomado astrônomo, astrofísico, cosmólogo, escritor e divulgador científico americano, famoso por popularizar a ciência e a exploração espacial através de seus livros, programas de TV e palestras, e por seu importante papel na missão Voyager e na busca por vida extraterrestre. Ele também foi um defensor da educação científica e do pensamento crítico.


In [5]:
response2 = openai.invoke(prompt)
print("Segunda resposta (usando cache):", response2)

Segunda resposta (usando cache): 

Carl Sagan foi um renomado astrônomo, astrofísico, cosmólogo, escritor e divulgador científico americano, famoso por popularizar a ciência e a exploração espacial através de seus livros, programas de TV e palestras, e por seu importante papel na missão Voyager e na busca por vida extraterrestre. Ele também foi um defensor da educação científica e do pensamento crítico.


# Disco / Banco de Dados

In [6]:
set_llm_cache(SQLiteCache(database_path="openai_cache.db"))

In [7]:
prompt = 'Me diga em poucas palavras quem foi Neil Armstrong.'

In [8]:
response1 = openai.invoke(prompt)
print("Primeira resposta (API chamada):", response1)

Primeira resposta (API chamada): 

Neil Armstrong foi um astronauta norte-americano, famoso por ter sido o primeiro homem a pisar na Lua durante a missão Apollo 11 em 1969. Ele foi um pioneiro da exploração espacial e um herói nacional nos Estados Unidos.


In [9]:
response2 = openai.invoke(prompt)
print("Segunda resposta (usando cache):", response2)

Segunda resposta (usando cache): 

Neil Armstrong foi um astronauta norte-americano, famoso por ter sido o primeiro homem a pisar na Lua durante a missão Apollo 11 em 1969. Ele foi um pioneiro da exploração espacial e um herói nacional nos Estados Unidos.


# Personalizado

In [10]:
class SimpleDiskCache:
    def __init__(self, cache_dir='cache_dir'):
        self.cache_dir = cache_dir
        os.makedirs(self.cache_dir, exist_ok=True)

    def _get_cache_path(self, key):
        hashed_key = hashlib.md5(key.encode()).hexdigest() #hasg cria nome de arquivo único
        return os.path.join(self.cache_dir, f"{hashed_key}.json")

    def lookup(self, key, llm_string):
        cache_path = self._get_cache_path(key)
        if os.path.exists(cache_path):
            with open(cache_path, 'r') as f:
                return json.load(f)
        return None

    def update(self, key, value, llm_string):
        cache_path = self._get_cache_path(key)
        with open(cache_path, 'w') as f:
            json.dump(value, f)

In [11]:
cache = SimpleDiskCache()
set_llm_cache(cache)
prompt = 'Me diga em poucas palavras quem foi Neil Degrasse Tyson.'

In [12]:
def invoke_with_cache(llm, prompt, cache):
    cached_response = cache.lookup(prompt, "")
    if cached_response:
        print("Usando cache:")
        return cached_response

    response = llm.invoke(prompt)
    cache.update(prompt, response, "")
    return response

In [13]:
response1 = invoke_with_cache(openai, prompt, cache)
response_text1 = response1.replace('\n', ' ') 

print("Primeira resposta (API chamada):", response_text1)

Primeira resposta (API chamada):   Neil deGrasse Tyson é um astrofísico, cosmólogo, escritor e divulgador científico americano, conhecido por popularizar a ciência e a astronomia através de palestras, livros e programas de TV. Ele também é o diretor do Planetário Hayden e um dos cientistas mais influentes do mundo. 


In [14]:
response2 = invoke_with_cache(openai, prompt, cache)
response_text2 = response2.replace('\n', ' ')  

print("Segunda resposta (usando cache):", response_text2)

Usando cache:
Segunda resposta (usando cache):   Neil deGrasse Tyson é um astrofísico, cosmólogo, escritor e divulgador científico americano, conhecido por popularizar a ciência e a astronomia através de palestras, livros e programas de TV. Ele também é o diretor do Planetário Hayden e um dos cientistas mais influentes do mundo. 
